In [ ]:
!pip install -r C:\Users\horne\rap_botV5\requirements.txt
# Evolution Experiments - evo_rhyme
# Cell 1: Imports
import sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import evo_rhyme
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline

  Using cached sentencepiece-0.2.1-cp312-cp312-win_amd64.whl.metadata (10 kB)
  Using cached scikit_learn-1.8.0-cp312-cp312-win_amd64.whl.metadata (11 kB)
  Using cached gensim-4.4.0-cp312-cp312-win_amd64.whl.metadata (8.6 kB)
  Using cached pronouncing-0.2.0-py2.py3-none-any.whl
  Using cached fasttext-0.9.3.tar.gz (73 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached pyyaml-6.0.3-cp312-cp312-win_amd64.whl.metadata (2.4 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
  U

  error: subprocess-exited-with-error
  
  × Building wheel for fasttext (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [136 lines of output]
      C:\Users\horne\AppData\Local\Temp\pip-build-env-ht_6nqm2\overlay\Lib\site-packages\setuptools\dist.py:599: SetuptoolsDeprecationWarning: Invalid dash-separated key 'description-file' in 'metadata' (setup.cfg), please use the underscore name 'description_file' instead.
      !!
      
              ********************************************************************************
              Usage of dash-separated 'description-file' will not be supported in future
              versions. Please use the underscore name 'description_file' instead.
              (Affected: fasttext).
      
              Available configuration options are listed in:
              https://setuptools.pypa.io/en/latest/userguide/declarative_config.html
      
              This deprecation is overdue, please update your project and remove depr

ModuleNotFoundError: No module named 'pronouncing'

In [ ]:
# Cell 2: Run evolution with --runs-dir to produce score_history.csv and top_candidates.json
import subprocess

result = subprocess.run(
    [
        sys.executable,
        str(ROOT / "scripts" / "run_couplet_evolution.py"),
        "--runs-dir",
        "--population", "50",
        "--generations", "15",
    ],
    cwd=str(ROOT),
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.stderr:
    print(result.stderr)
print(f"Exit code: {result.returncode}")

# Find the most recent run directory (with score_history.csv)
runs_dir = ROOT / "data" / "evo_rhyme" / "runs"
run_dirs = sorted(
    (d for d in runs_dir.glob("*") if d.is_dir() and (d / "score_history.csv").exists()),
    key=lambda p: p.stat().st_mtime,
    reverse=True,
)
run_dir = run_dirs[0] if run_dirs else None
print(f"\nRun dir: {run_dir}")

In [ ]:
# Cell 3: Load score_history.csv and plot fitness over generations (best, avg)
if run_dir is None:
    runs_dir = ROOT / "data" / "evo_rhyme" / "runs"
    run_dirs = sorted(
        (d for d in runs_dir.glob("*") if d.is_dir() and (d / "score_history.csv").exists()),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    run_dir = run_dirs[0] if run_dirs else None
if run_dir is None:
    raise FileNotFoundError("No run directory found. Run cell 2 first.")
df = pd.read_csv(run_dir / "score_history.csv")

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(df["gen"], df["best_fitness"], label="Best fitness", marker="o", markersize=4)
ax.plot(df["gen"], df["avg_fitness"], label="Avg fitness", marker="s", markersize=4)
ax.set_xlabel("Generation")
ax.set_ylabel("Fitness")
ax.set_title("Fitness over Generations")
ax.legend()
ax.grid(True, alpha=0.3)
fig.savefig(run_dir / "fitness_over_generations.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Cell 4: Plot diversity and acceptance_rate over generations
fig, ax1 = plt.subplots(figsize=(8, 5))
ax1.plot(df["gen"], df["diversity"], color="C0", label="Diversity", marker="o", markersize=4)
ax1.set_xlabel("Generation")
ax1.set_ylabel("Diversity", color="C0")
ax1.tick_params(axis="y", labelcolor="C0")
ax1.legend(loc="upper left")
ax1.grid(True, alpha=0.3)

ax2 = ax1.twinx()
ax2.plot(df["gen"], df["acceptance_rate"], color="C1", label="Acceptance rate", marker="s", markersize=4)
ax2.set_ylabel("Acceptance rate", color="C1")
ax2.tick_params(axis="y", labelcolor="C1")
ax2.legend(loc="upper right")

ax1.set_title("Diversity and Acceptance Rate over Generations")
fig.tight_layout()
fig.savefig(run_dir / "diversity_acceptance.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Cell 5: Load top_candidates.json and show rhyme-family distribution (bar chart of end-tail frequencies)
import json
from collections import Counter
from evo_rhyme.phonetics import extract_rhyme_tail, tokenize_line

with open(run_dir / "top_candidates.json", encoding="utf-8") as f:
    top_candidates = json.load(f)

# Flatten all candidates and extract end-tail from each line's last word
# Supports couplet format (line1, line2) and verse format (lines)
tails = []
for gen_key, candidates in top_candidates.items():
    for c in candidates:
        lines = [c["line1"], c["line2"]] if "line1" in c and "line2" in c else c.get("lines", [])
        for line in lines:
            tokens = tokenize_line(line) if isinstance(line, str) else []
            if tokens:
                tail = extract_rhyme_tail(tokens[-1].lower())
                if tail:
                    tails.append(tail)

tail_counts = Counter(tails)
labels = list(tail_counts.keys())
counts = list(tail_counts.values())

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(range(len(labels)), counts, color="steelblue", edgecolor="navy", alpha=0.8)
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=45, ha="right")
ax.set_xlabel("End-tail (rhyme family)")
ax.set_ylabel("Frequency")
ax.set_title("Rhyme-Family Distribution (End-Tail Frequencies)")
fig.tight_layout()
fig.savefig(run_dir / "rhyme_family_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Cell 6: Component score breakdown for top candidate (bar chart: end_rhyme, internal, syllable, semantic, etc.)
last_gen = max(top_candidates.keys(), key=int)
top_candidate = top_candidates[last_gen][0]
scores = top_candidate.get("scores", {})

all_keys = list(scores.keys())
values = [scores[k] for k in all_keys]

fig, ax = plt.subplots(figsize=(10, 5))
colors = ["steelblue" if v >= 0 else "coral" for v in values]
ax.bar(range(len(all_keys)), values, color=colors, edgecolor="black", alpha=0.8)
ax.set_xticks(range(len(all_keys)))
ax.set_xticklabels(all_keys, rotation=45, ha="right")
ax.set_xlabel("Component")
ax.set_ylabel("Score")
l1 = top_candidate.get("line1", top_candidate.get("lines", [""])[0] if top_candidate.get("lines") else "")
l2 = top_candidate.get("line2", top_candidate.get("lines", ["", ""])[1] if len(top_candidate.get("lines", [])) > 1 else "")
preview = f"\"{l1[:50]}{'...' if len(l1) > 50 else ''}\" | \"{l2[:50]}{'...' if len(l2) > 50 else ''}\""
ax.set_title(f"Component Score Breakdown - Top Candidate\n{preview}")
ax.axhline(y=0, color="gray", linestyle="-", linewidth=0.5)
fig.tight_layout()
fig.savefig(run_dir / "component_score_breakdown.png", dpi=150, bbox_inches="tight")
plt.show()